In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------
# Config
# ------------------------------
SCORES_CSV = Path("dataset_scores_table.csv")

CATS = [
    "Attack Coverage",
    "Provence Score",
    "Feature Diversity",
    "Documentation & Metadata",
    "Evaluation Readiness",
    "Availability",
]

# ------------------------------
# Helpers
# ------------------------------
def weighted_composite(df: pd.DataFrame, weights: dict) -> pd.Series:
    w = np.array([weights[c] for c in CATS], dtype=float)
    w = w / w.sum()
    return pd.Series((df[CATS].values * w).sum(axis=1), index=df.index)

def spearman_rho_from_ranks(rank_a: pd.Series, rank_b: pd.Series) -> float:
    a = rank_a.to_numpy(dtype=float)
    b = rank_b.to_numpy(dtype=float)
    a_mean = a.mean()
    b_mean = b.mean()
    num = ((a - a_mean) * (b - b_mean)).sum()
    den = np.sqrt(((a - a_mean) ** 2).sum() * ((b - b_mean) ** 2).sum())
    return float(num / den) if den != 0 else float("nan")

# ------------------------------
# Load
# ------------------------------
scores = pd.read_csv(SCORES_CSV).set_index("Dataset")

# ------------------------------
# Weight schemes
# ------------------------------
W_EQUAL = {c: 1 / len(CATS) for c in CATS}

# provenance emphasis: provenance=0.30, remaining share 0.70 equally
W_PROV = {c: 0.0 for c in CATS}
W_PROV["Provence Score"] = 0.30
for c in CATS:
    if c != "Provence Score":
        W_PROV[c] = (1.0 - 0.30) / (len(CATS) - 1)

# evaluation readiness emphasis: eval=0.30, remaining share 0.70 equally
W_EVAL = {c: 0.0 for c in CATS}
W_EVAL["Evaluation Readiness"] = 0.30
for c in CATS:
    if c != "Evaluation Readiness":
        W_EVAL[c] = (1.0 - 0.30) / (len(CATS) - 1)

# ------------------------------
# Compute composites
# ------------------------------
comp_equal = weighted_composite(scores, W_EQUAL).rename("Equal")
comp_prov  = weighted_composite(scores, W_PROV).rename("ProvenanceEmphasis")
comp_eval  = weighted_composite(scores, W_EVAL).rename("EvalReadinessEmphasis")

composites = pd.concat([comp_equal, comp_prov, comp_eval], axis=1)

# Ranks: 1 = best. method="min" keeps ties consistent and journal-friendly.
ranks = composites.rank(ascending=False, method="min").astype(int)

# Spearman rho between rank vectors
rho_eq_prov = spearman_rho_from_ranks(ranks["Equal"], ranks["ProvenanceEmphasis"])
rho_eq_eval = spearman_rho_from_ranks(ranks["Equal"], ranks["EvalReadinessEmphasis"])

print("N datasets:", len(ranks))
print("Spearman rho (Equal vs ProvenanceEmphasis):", round(rho_eq_prov, 4))
print("Spearman rho (Equal vs EvalReadinessEmphasis):", round(rho_eq_eval, 4))

# ------------------------------
# Save appendix-friendly tables
# ------------------------------
weights_table = pd.DataFrame(
    [
        ["Equal", *[W_EQUAL[c] for c in CATS]],
        ["Provenance-emphasis", *[W_PROV[c] for c in CATS]],
        ["Evaluation-readiness emphasis", *[W_EVAL[c] for c in CATS]],
    ],
    columns=["Scheme", *CATS],
)

weights_table.to_csv("appendix_weight_vectors.csv", index=False)
ranks.to_csv("appendix_ranks_by_scheme.csv")

# Optional: save composite scores too
composites.to_csv("appendix_composites_by_scheme.csv")

N datasets: 13
Spearman rho (Equal vs ProvenanceEmphasis): 0.9666
Spearman rho (Equal vs EvalReadinessEmphasis): 0.9669
